# Лабораторна 3. Токенізація та трансформери

У цій лабораторній роботі ви пройдете шлях від механіки токенізації до практичного використання готових мовних моделей:

- алгоритми токенізації (BPE, WordPiece, Unigram) — натренуєте всі три на власному тексті;
- mini-GPT з нуля — повторите тренування трансформера на своєму датасеті;
- готові моделі для різних NLP-задач — класифікація, генерація, переклад, Q&A та інше через `pipeline()`;
- порівняння кількох моделей на одній задачі;
- творчий застосунок на основі готової моделі.

Демо-ноутбук лекції 4 (`transformers_demo.ipynb`) — основа для секцій 2-3, звертайтесь до нього як до довідника. Він лежить в архіві демо-матеріалів лекції 4 — посилання «Демо-матеріали (zip)» є в картці лекції на головній сторінці курсу. Розділи 1 і 4 спираються на офіційний [Hugging Face LLM Course](https://huggingface.co/learn/llm-course/en/chapter1/1) — читайте відповідні розділи курсу перед виконанням, посилання додані в кожній секції.

## Матеріали

- [3Blue1Brown — Transformers, the tech behind LLMs](https://www.youtube.com/watch?v=wjZofJX0v4M)
- [3Blue1Brown — Attention in transformers, step-by-step](https://www.youtube.com/watch?v=eMlx5fFNoYc)
- [3Blue1Brown — How might LLMs store facts](https://www.youtube.com/watch?v=9-Jl0dxWQs8)
- [Hugging Face LLM Course — Chapter 1: Transformer Models](https://huggingface.co/learn/llm-course/en/chapter1/1)
- [Hugging Face LLM Course — Chapter 2: Using Transformers](https://huggingface.co/learn/llm-course/en/chapter2/1)
- [Hugging Face LLM Course — Chapter 6: The Tokenizers library](https://huggingface.co/learn/llm-course/en/chapter6/1)

In [2]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    LAB_DIR = "/content/drive/MyDrive/lab3"
else:
    LAB_DIR = os.path.abspath(".")

MODELS_DIR = os.path.join(LAB_DIR, "models")
RUNS_DIR = os.path.join(LAB_DIR, "runs")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)

print("Артефакти зберігаються в:", LAB_DIR)

Mounted at /content/drive
Артефакти зберігаються в: /content/drive/MyDrive/lab3


## 0. Опрацюйте демо-ноутбук лекції 4

Прогляньте пояснення і запустіть усі клітинки демо-ноутбука.

Після цього дайте відповідь на запитання нижче.

### 0.1. Теоретичні питання (якщо потрібно дописати якийсь код, то в блоці "Мої відповіді наведіть відповідний код")

1. У розділі 2.1 демо тренер BPE отримує список окремих віршів, а не один склеєний рядок. Що б змінилось (і що б зламалось), якби ми передали один суцільний рядок?
2. З який етапів складається токенізація, поясніть сутність кожного з етапів (див. [туторіал по токенізації](https://huggingface.co/learn/llm-course/en/chapter6/8))
2. Що таке `"<unk>", "<eos>"` в 2.1.?
3. Зменшіть `VOCAB_SIZE` (2.1) удвічі-втричі (наприклад з 16000 до 4000-6000) і перетренуйте BPE-токенізатор на тому самому корпусі. Порівняйте середню кількість токенів на вірш і частку `<unk>` до/після зміни. Поясніть, чому менший словник не підвищує `<unk>`, але впливає на довжину закодованих послідовностей (а отже — і на кількість кроків, які "бачить" модель за той самий `CONTEXT_LEN`).
4. У розділі 2.2 між віршами вставлено токен `<eos>`. Навіщо? Що конкретно піде не так під час тренування mini-GPT, якщо цей роздільник прибрати?
5. Який формат мають і що репрезентують елементи тензорів `train_data` & `val_data` у розділі 2.3?
6. Допишіть в 3.2. `assert` тест, який перевіряє відповідність частин випадкового батчу.
7. У 3.3 `causal_mask` — нижньотрикутна матриця одиниць. Чому саме нижньотрикутна, а не верхньотрикутна? Що б змінилося в поведінці моделі при генерації тексту, якби маски не було?
8. У 3.10 генерація бере чекпоінт із найкращим `val_loss`, а не останній. За яких умов ці два чекпоінти збігаються, а за яких — ні? Подивіться на свою криву навчання (3.9) і скажіть, який випадок у вас.
10. Зменшіть `EVAL_INTERVAL` у 5 разів (наприклад з 250 до 50) і повторно запустіть тренування. Чи стала крива навчання (3.9) помітно гладкішою/інформативнішою? Скільки додаткового часу це коштує (частіші виклики `estimate_loss()` на `EVAL_ITERS` батчах), і чи є сенс робити `EVAL_INTERVAL` ще меншим?
11. У 4.2 той самий промпт дає дуже різний текст при `temperature=0.2` і `temperature=1.4`. Поясніть математично, що робить `temperature` з логітами перед `softmax`, і чому при дуже високій температурі текст стає майже випадковим.
12. За що саме відповідає параметр `top_p`?
13. У 4.2 змініть налаштування `enable_thinking=False` на `True` запустіть повторно клітинку і прокоментуйте зміни у результаті виконання.

### 0.2. Мої відповіді:
1. Якщо передати один величезний зліплений текст, комп'ютеру доведеться тримати весь цей блок у пам'яті одночасно під час підрахунку частот, що може призвести до нестачі оперативної пам'яті. Також, передавати текст частинами є природнішим та легшим способом для алгоритму тренування.
2. Процес токенізації починається з нормалізації, очищення тексту, тоді коли видаляються зайві пробіли, дивні символи, і всі літери переводяться в нижній регістр. Далі настає черга пре-токенізації, яка розбиває текст на базові шматки просто по пробілах або розділових знаках, щоб алгоритм випадково не зліпив кінець одного слова з початком іншого. Після цього вмикається сама модель, наприклад BPE, і відбувається процес, коли алгоритм об'єднує найчастіші комбінації букв у єдині токени або підслова. Завершується все пост-процесингом, коли до готового результату додаються спеціальні службові маркери початку або кінця тексту, щоб модель правильно розуміла загальну структуру.
3.
*    `"<unk>"` (unknown) - це токен для "невідомих" слів або символів, яких немає у словнику токенізатора.  
*  `"<eos>"` (end of sentence/sequence) — це спеціальний маркер, який показує, що поточний вірш чи текст закінчився і далі піде наступний.
4. Кількість невідомих слів не збільшиться, тому що BPE-токенізатор знає всі базові букви і почне розбивати великі слова на склади чи окремі літери. Але через це одне слово тепер складатиметься з 3-5 токенів замість одного. Тому, довжина тексту в токенах сильно виросте, і в те саме вікно контексту влізе набагато менше реальних слів, бо модель бачитиме їх по шматках.
5. Він потрібен, щоб позначити межу між окремими текстами. Якщо його прибрати, то останнє слово одного вірша зліпиться з першим словом наступного. Через це модель подумає, що це один суцільний текст і почне вчитися нелогічних переходів.
6. Це величезні одновимірні списки чисел, тензори у форматі int64. Ці числа просто ID-номери токенів, у які закодовано весь текстовий корпус. Вони репрезентують весь текст, готовий для того, щоб його згодувати нейромережі.
7. Цей код бере всі токени з xb, починаючи з другого, і порівнює їх з усіма токенами yb, окрім останнього. Так як yb це той самий текст, просто посунутий на одне слово вперед, тому ці шматки мають збігатися.
```
assert torch.all(xb[:, 1:] == yb[:, :-1]), "Помилка: таргет y не є зсунутою версією x!"
```
8. Нижньотрикутна форма дозволяє токену дивитися лише на себе та на попередні слова. Якби використовували верхньотрикутну маску, то модель дивилася б в майбутнє. Якби маски взагалі не було, то модель під час навчання просто бачила б наступне слово наперед замість того, щоб вчитися його вгадувати.
9. Вони збігаються, якщо модель ставала все розумнішою і розумнішою, типу коли помилка на валідації падала, аж до самої останньої ітерації. Вони відрізняються, якщо десь посеред тренування модель почала перенавчатися, тоді тренувальна помилка далі падає, а валідаційна починає рости.
10. Якщо перевіряти модель частіше, графік навчання стане набагато детальнішим і плавнішим, і це дозволить фіксувати найменші коливання. Але це вимагатиме  більше часу, так як на кожну перевірку витрачаються додаткові ресурси процесора/відеокарти. Робити інтервал занадто малим немає сенсу, тому що це затягне тренування, а практична цінність від такого детального графіка буде незначною.
11. Температура — це число, на яке діляться початкові оцінки моделі перед тим, як перетворити їх на ймовірності. Якщо поділити ці оцінки на велике число (наприклад, 1.4), вони стануть приблизно однаковими, через що ймовірності вирівнюються, а згенерований текст стане дуже випадковим і може втратити логічний сенс. Якщо ж поділити на маленьке число (наприклад, 0.2), розрив між найімовірнішим словом і всіма іншими стає великим, тому модель уникає будь-яких відхилень і видає лише найбільш передбачуваний, шаблонний текст.
12. top_p - це параметр, який обмежує вибір найбільш імовірними словами. Він відкидає слова з дуже низькою ймовірністю. Модель бере лише найкращі варіанти, сумарна ймовірність яких досягає відсотка p, а всі менш імовірні слова просто ігнорує.
13. Цей параметр змушує модель спочатку згенерувати свої проміжні роздуми. Модель почне виводити текст про те, як саме вона будує логіку перед тим, як згенерувати фінальний результат.

---
## 1. Свій датасет

Знайдіть **свій** українськомовний текстовий корпус, достатній за обсягом для mini-GPT — наприклад, на [Hugging Face Datasets](https://huggingface.co/datasets?language=language:uk) або на [Kaggle](https://www.kaggle.com/datasets?search=ukrainian). Жанр і епоха — на ваш вибір (новини, художня література, вікіпедія, форуми, судові рішення тощо), головне — одна мова (українська) і достатньо тексту, щоб токенізатор і модель побачили бодай прості закономірності.

⚠️ Якщо датасет великий (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`), інакше отримаєте помилку нестачі пам'яті (out of memory, OOM).

⚠️ Якщо датасет доступний лише через `streaming=True` (наприклад, через розмір) — `load_dataset(..., streaming=True).take(N)` повертає `IterableDataset`, у якого **нема** `.to_pandas()` і `len()`. Спершу матеріалізуйте вибірку в список, а вже потім — у `DataFrame`:
```python
rows = list(streamed_ds)
poems_df = pd.DataFrame(rows)
```

In [3]:
from datasets import load_dataset
import pandas as pd

wiki_dataset = load_dataset("wikimedia/wikipedia", "20231101.uk", split="train[:5000]")

wiki_df = wiki_dataset.to_pandas()

corpus_text = "\n\n".join(wiki_df["text"].dropna().tolist())

print(f"Усього статей: {len(wiki_df)}")
print(f"Загальний розмір корпусу: {len(corpus_text)} символів")
print("\n--- Приклад тексту (перші 500 символів) ---\n")
print(corpus_text[:500])

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

20231101.uk/train-00000-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  382MB            

20231101.uk/train-00000-of-00010.parquet: downloading bytes:           |  0.00B            

20231101.uk/train-00001-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  230MB            

20231101.uk/train-00001-of-00010.parquet: downloading bytes:           |  0.00B            

20231101.uk/train-00002-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  182MB            

... (13 outputs left out)

20231101.uk/train-00009-of-00010.parquet: reconstructing file:   0%|          |  0.00B /  197MB            

20231101.uk/train-00009-of-00010.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1294720 [00:00<?, ? examples/s]

Усього статей: 5000
Загальний розмір корпусу: 33090135 символів

--- Приклад тексту (перші 500 символів) ---

Геогра́фія, або заст. земле́пис (від  — опис Землі; де  — Земля і  — писати, описувати) — система наук, що вивчає географічну оболонку Землі (епігеосферу), її просторову природну і соціально-економічну різноманітність, господарство і населення планети, окремих її регіонів та країн, а також зв'язки між природним середовищем і діяльністю людини. В сучасному розумінні поняття «географія» заміщено поняттям «географічні науки».

Поняття 
Уперше термін «географія» був запропонований давньогрецьким гео


In [4]:
words = corpus_text.split()
unique_words = set(w.lower() for w in words)

print("Слів (з повторами):   ", len(words))
print("Унікальних слів:      ", len(unique_words))

Слів (з повторами):    4465195
Унікальних слів:       518596


---
## 2. Токенізатори: BPE, WordPiece, Unigram

Демо тренує лише BPE. У цій секції ви натренуєте й порівняєте всі три основні алгоритми токенізації на **своєму** тексті.

Матеріал — офіційний [Hugging Face LLM Course, розділ 6](https://huggingface.co/learn/llm-course/en/chapter6/1). Пройдіть і повторіть на своїх даних:

1. **[6.2. Тренування нового токенізатора на основі старого](https://huggingface.co/learn/llm-course/en/chapter6/2)** — найпростіший шлях: `tokenizer.train_new_from_iterator(...)` на основі архітектури готового токенізатора (наприклад, `Qwen/Qwen2.5-0.5B-Instruct`). Зробіть це як розминку перед ручними варіантами нижче.
2. **[6.5. Алгоритм BPE](https://huggingface.co/learn/llm-course/en/chapter6/5)** — повторіть покрокову реалізацію з туторіалу на своєму тексті, повністю вручну або за допомогою бібліотеки `tokenizers`.
3. **[6.6. Алгоритм WordPiece](https://huggingface.co/learn/llm-course/en/chapter6/6)** — так само.
4. **[6.7. Алгоритм Unigram](https://huggingface.co/learn/llm-course/en/chapter6/7)** — так само.

### 1.1. Порівняння

Для однакового `vocab_size` (візьміть, наприклад, 500 — на невеликому зразку тексту більший словник не встигне показати різницю) порівняйте три токенізатори в невеликій таблиці: розмір словника, кількість токенів на вашому зразку, скільки токенів виходить на кількох обраних словах (частотне й рідкісне — як у лекції). Коротко письмово: який алгоритм дав найкоротші послідовності на вашому тексті і чи це узгоджується з тим, що каже туторіал про природу кожного алгоритму?

In [5]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
import pandas as pd

VOCAB_SIZE = 500
SPECIAL_TOKENS = ["<unk>", "<eos>"]

train_iterator = wiki_df["text"].dropna().tolist()[:500]

bpe_tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
bpe_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe_tokenizer.decoder = decoders.ByteLevel()
bpe_trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS)
bpe_tokenizer.train_from_iterator(train_iterator, trainer=bpe_trainer)

wp_tokenizer = Tokenizer(models.WordPiece(unk_token="<unk>"))
wp_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
wp_trainer = trainers.WordPieceTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS, limit_alphabet=100)
wp_tokenizer.train_from_iterator(train_iterator, trainer=wp_trainer)

uni_tokenizer = Tokenizer(models.Unigram())
uni_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
uni_trainer = trainers.UnigramTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS, unk_token="<unk>")
uni_tokenizer.train_from_iterator(train_iterator, trainer=uni_trainer)

sample_text = "Українська вікіпедія містить багато цікавих і корисних статей про науку."
test_words = ["вікіпедія", "про", "антропоцентризм"]

results = []
for name, tok in [("BPE", bpe_tokenizer), ("WordPiece", wp_tokenizer), ("Unigram", uni_tokenizer)]:
    encoded_sample = tok.encode(sample_text).tokens

    word_tokens = {word: tok.encode(word).tokens for word in test_words}

    results.append({
        "Алгоритм": name,
        "Розмір словника": tok.get_vocab_size(),
        "Токенів у тестовому реченні": len(encoded_sample),
        f"Токени для '{test_words[0]}'": len(word_tokens[test_words[0]]),
        f"Токени для '{test_words[1]}'": len(word_tokens[test_words[1]]),
        f"Токени для '{test_words[2]}'": len(word_tokens[test_words[2]]),
    })

comparison_df = pd.DataFrame(results)
print("Порівняння токенізаторів")
display(comparison_df)

Порівняння токенізаторів


,Алгоритм,Розмір словника,Токенів у тестовому реченні,Токени для 'вікіпедія',Токени для 'про',Токени для 'антропоцентризм'
0,BPE,500,36,6,3,10
1,WordPiece,500,35,7,1,10
2,Unigram,500,44,8,1,13


У першому запуску WordPiece дав словник 770 замість 500 і 63 токени на речення, тобто різав текст по одній літері. Початковий алфавіт (у вікі багато рідкісних символів) разом з копіями ## зайняв увесь ліміт, і на злиття місця не лишилось. Після limit_alphabet=100 словник став 500, і порівняння стало чесним.

Тепер найкоротшу послідовність дав WordPiece (35 токенів), майже стільки ж BPE (36), а найдовшу Unigram (44). Слово «про» WordPiece і Unigram зберегли цілим (1 токен). BPE розбив його на 3, бо працює на рівні байтів і часте слово в нього записане разом із пробілом попереду (« про»). Окреме «про» без пробілу для нього рідкісне. Рідкісне слово «антропоцентризм» усі алгоритми розбили на багато шматків: 10, 10 і 13 токенів.

Це узгоджується з туторіалом. BPE і WordPiece будують словник знизу вгору, зливаючи пари символів у довші підслова, тому дають короткі послідовності. Unigram іде згори вниз: відкидає зайві токени з великого словника. При малому словнику (500) у нього лишається найбільше дрібних шматків, хоча частотні слова він зберігає цілими.

---
## 3. mini-GPT на своїх даних

Повторіть підхід із розділів 2-3 демо-ноутбука на своєму датасеті — не копіюйте код один в один: у вашого датасету інша колонка з текстом, інший обсяг, інший жанр/регістр мови — токенізатор і модель побачать зовсім інші закономірності, ніж на поезії з демо.

### 3.1. Корпус і токенізація

1. Завантажте свій датасет і зберіть `corpus_text` з потрібної колонки (демо 1.2).
2. Натренуйте токенізатор на своєму корпусі (демо 2.1). **Якщо ваш датасет великий** (сотні МБ тексту і більше) — кодуйте корпус пачками, як у демо 2.2 (`encode_batch` + конвертація в `numpy.int32`). Не забудьте `<eos>` між документами (демо 2.2).
3. Спліт train/val (демо 2.3).

### 3.2. Модель: щонайменше 2 варіанти

За зразком розділу 3 демо (`AttentionHead`, `MultiHeadAttention`, `TransformerBlock`, `MiniGPT`) — тренуйте щонайменше 2 варіанти, які відрізняються параметрами: інший `VOCAB_SIZE`, інша `CONTEXT_LEN`, інша кількість шарів/голів (`N_LAYERS`/`N_HEADS`), або інший `MAX_ITERS`. Кожен варіант логуйте в TensorBoard в окрему підпапку (як у демо 3.8), з осмисленою назвою.

### 3.3. Генерація і висновки

Для кожного варіанта — генерація на найкращому за `val_loss` чекпоінті (демо 3.10), той самий промпт для всіх варіантів, щоб порівняння було чесним. Коротко письмово: який варіант дав кращий `val_loss` і чи помітно це за якістю згенерованого тексту? Що саме (зміна словника, глибини мережі, довжини контексту) дало найбільший ефект?

In [30]:
import gc
import numpy as np
import torch
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

VOCAB_SIZE = 5000
SPECIAL_TOKENS = ["<unk>", "<eos>"]

bpe_tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
bpe_tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
bpe_tokenizer.decoder = decoders.ByteLevel()

bpe_trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS)
wiki_iterator = wiki_df["text"].dropna().tolist()
bpe_tokenizer.train_from_iterator(wiki_iterator, trainer=bpe_trainer)

bpe_vocab = bpe_tokenizer.get_vocab()
itos = [None] * len(bpe_vocab)
for tok, idx in bpe_vocab.items():
    itos[idx] = tok
stoi = bpe_vocab
UNK_ID = stoi["<unk>"]
EOS_ID = stoi["<eos>"]

def encode(text):
    return [stoi.get(t, UNK_ID) for t in bpe_tokenizer.encode(text).tokens]

def decode(ids):
    return bpe_tokenizer.decode(ids)

ENCODE_BATCH_SIZE = 1000
token_id_chunks = []
for start in range(0, len(wiki_iterator), ENCODE_BATCH_SIZE):
    batch = wiki_iterator[start:start + ENCODE_BATCH_SIZE]
    encodings = bpe_tokenizer.encode_batch(batch)
    for enc in encodings:
        token_id_chunks.append(np.array(enc.ids + [EOS_ID], dtype=np.int32))

token_ids = np.concatenate(token_id_chunks)
data = torch.from_numpy(token_ids.astype(np.int64))

split_idx = int(len(data) * 0.9)
train_data = data[:split_idx]
val_data = data[split_idx:]

print(f"Дані розділено. Тренувальна вибірка має розмір {train_data.shape}, валідаційна {val_data.shape}")

Дані розділено. Тренувальна вибірка має розмір torch.Size([9840248]), валідаційна torch.Size([1093361])


In [31]:
import math
import time
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class AttentionHead(nn.Module):
    def __init__(self, embed_dim, head_dim, context_len, dropout):
        super().__init__()
        self.query = nn.Linear(embed_dim, head_dim, bias=False)
        self.key = nn.Linear(embed_dim, head_dim, bias=False)
        self.value = nn.Linear(embed_dim, head_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("causal_mask", torch.tril(torch.ones(context_len, context_len)))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.query(x), self.key(x), self.value(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        scores = scores.masked_fill(self.causal_mask[:T, :T] == 0, float("-inf"))
        weights = self.dropout(F.softmax(scores, dim=-1))
        return weights @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, n_heads, context_len, dropout):
        super().__init__()
        head_dim = embed_dim // n_heads
        self.heads = nn.ModuleList([AttentionHead(embed_dim, head_dim, context_len, dropout) for _ in range(n_heads)])
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.proj(torch.cat([h(x) for h in self.heads], dim=-1)))

class FeedForward(nn.Module):
    def __init__(self, embed_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.ReLU(),
            nn.Linear(4 * embed_dim, embed_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, n_heads, context_len, dropout):
        super().__init__()
        self.att = MultiHeadAttention(embed_dim, n_heads, context_len, dropout)
        self.ff = FeedForward(embed_dim, dropout)
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = x + self.att(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, embed_dim, n_heads, n_layers, context_len, dropout):
        super().__init__()
        self.context_len = context_len
        self.tok_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(context_len, embed_dim)
        self.blocks = nn.Sequential(*[TransformerBlock(embed_dim, n_heads, context_len, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.lm_head = nn.Linear(embed_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device))
        x = self.ln_f(self.blocks(x))
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.context_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, num_samples=1)], dim=1)
        self.train()
        return idx

In [32]:
def get_batch(split, context_len, batch_size):
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - context_len - 1, (batch_size,))
    x = torch.stack([source[i:i + context_len] for i in ix])
    y = torch.stack([source[i + 1:i + context_len + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_loss(model, context_len, batch_size, eval_iters=50):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for i in range(eval_iters):
            xb, yb = get_batch(split, context_len, batch_size)
            _, loss = model(xb, yb)
            losses[i] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

def train_model(run_name, embed_dim, n_heads, n_layers, context_len, max_iters=2500):
    print(f"\nПочинаємо тренування {run_name}")
    model = MiniGPT(VOCAB_SIZE, embed_dim, n_heads, n_layers, context_len, dropout=0.1).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

    ckpt_path = os.path.join(MODELS_DIR, f"{run_name}.pt")
    writer = SummaryWriter(log_dir=os.path.join(RUNS_DIR, run_name))

    best_val_loss = float("inf")
    batch_size = 64
    eval_interval = 250

    for it in range(max_iters):
        xb, yb = get_batch("train", context_len, batch_size)
        _, loss = model(xb, yb)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        if it % eval_interval == 0 or it == max_iters - 1:
            losses = estimate_loss(model, context_len, batch_size)
            writer.add_scalar("Loss/train", losses["train"], it)
            writer.add_scalar("Loss/val", losses["val"], it)
            print(f"Ітерація {it:4d} | Train Loss {losses['train']:.3f} | Val Loss {losses['val']:.3f}")

            if losses["val"] < best_val_loss:
                best_val_loss = losses["val"]
                torch.save(model.state_dict(), ckpt_path)

    writer.close()
    print(f"Завершено Найкращий чекпоінт збережено у {ckpt_path} (Val loss {best_val_loss:.3f})")
    return best_val_loss

# Варіант 1 Стандартний (4 шари, контекст 64)
loss_v1 = train_model("minigpt_v1", embed_dim=128, n_heads=4, n_layers=4, context_len=64)

# Варіант 2 Довгий контекст (2 шари, контекст 128)
loss_v2 = train_model("minigpt_v2", embed_dim=128, n_heads=4, n_layers=2, context_len=128)


Починаємо тренування minigpt_v1
Ітерація    0 | Train Loss 8.628 | Val Loss 8.634
Ітерація  250 | Train Loss 6.984 | Val Loss 6.983
Ітерація  500 | Train Loss 6.542 | Val Loss 6.589
Ітерація  750 | Train Loss 6.092 | Val Loss 6.216
Ітерація 1000 | Train Loss 5.772 | Val Loss 5.949
Ітерація 1250 | Train Loss 5.534 | Val Loss 5.748
Ітерація 1500 | Train Loss 5.318 | Val Loss 5.585
Ітерація 1750 | Train Loss 5.148 | Val Loss 5.444
Ітерація 2000 | Train Loss 5.045 | Val Loss 5.349
Ітерація 2250 | Train Loss 4.944 | Val Loss 5.236
Ітерація 2499 | Train Loss 4.853 | Val Loss 5.203
Завершено Найкращий чекпоінт збережено у /content/drive/MyDrive/lab3/models/minigpt_v1.pt (Val loss 5.203)

Починаємо тренування minigpt_v2
Ітерація    0 | Train Loss 8.652 | Val Loss 8.652
Ітерація  250 | Train Loss 6.977 | Val Loss 6.981
Ітерація  500 | Train Loss 6.552 | Val Loss 6.621
Ітерація  750 | Train Loss 6.087 | Val Loss 6.218
Ітерація 1000 | Train Loss 5.770 | Val Loss 5.968
Ітерація 1250 | Train Loss 

In [10]:
def generate_text(model_path, prompt, embed_dim, n_heads, n_layers, context_len, max_new_tokens=40):
    model = MiniGPT(VOCAB_SIZE, embed_dim, n_heads, n_layers, context_len, dropout=0.1).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))

    prompt_ids = torch.tensor([encode(prompt)], dtype=torch.long).to(device)
    out_ids = model.generate(prompt_ids, max_new_tokens, temperature=0.8, top_k=40)
    return decode(out_ids[0].tolist())

start_word = "Україна "

print("Варіант 1 (4 шари, контекст 64)")
print(generate_text(os.path.join(MODELS_DIR, "minigpt_v1.pt"), start_word, 128, 4, 4, 64))

print("\nВаріант 2 (2 шари, контекст 128)")
print(generate_text(os.path.join(MODELS_DIR, "minigpt_v2.pt"), start_word, 128, 4, 2, 128))

Варіант 1 (4 шари, контекст 64)


NameError: name 'MiniGPT' is not defined

За результатами тренування перший варіант (4 шари, контекст 64) показав кращий val_loss 5.203 проти 5.296 у другого (2 шари, контекст 128). Однак варіанти відрізняються одразу двома параметрами, тому не можна точно сказати, що саме дало ефект: більша глибина чи коротший контекст. Різниця невелика (~0.09), а val_loss в обох ще падав на останній ітерації, тобто моделі недонавчені, і найкращий чекпоінт збігається з останнім. За текстом різниця теж помітна слабо: обидва варіанти видають уривки, схожі на вікі-статті, але без зв'язного змісту.


---
## 4. Playground готових моделей

Тур по задачах через `pipeline()` — на основі [HF LLM Course, Chapter 1, §3](https://huggingface.co/learn/llm-course/en/chapter1/3).

_Зверніть увагу_: в межах попередніх завдань Ви могли завантажити памʼять "під завʼязку", тому для безпроблемного виконання наступних секцій вивільніть памʼять:

1. **Спершу збережіть усе потрібне на диск** — чекпоінти mini-GPT (`models/minigpt_v*.pt`), токенізатори, `results.csv`. Секції 4-6 від змінних секцій 1-3 не залежать, тож після цього їх можна сміливо видаляти.
2. **Видаліть великі обʼєкти** — моделі, оптимізатори, тензори `train_data`/`val_data`, корпус і `DataFrame`-и — і зберіть сміття:
   ```python
   import gc, torch

   del model, optimizer, train_data, val_data, corpus_text  # підставте свої імена змінних
   gc.collect()
   if torch.cuda.is_available():
       torch.cuda.empty_cache()
       print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")
   ```
   Памʼять звільниться лише тоді, коли на обʼєкт не лишиться **жодного** посилання: зверніть увагу на копії в інших змінних, списках і словниках (наприклад, словник із кількома варіантами моделей).
3. **Перевірте, що памʼять справді звільнилась**: індикатор RAM/GPU вгорі праворуч.
4. **У самій секції 4 не тримайте всі pipeline-и одночасно**: кожен `pipeline(...)` завантажує окрему модель. Перевикористовуйте одну змінну (як `classifier` нижче) або робіть `del` + `gc.collect()` після кожної задачі.
5. **Стежте за диском**: моделі кешуються в `~/.cache/huggingface/hub`, і кілька моделей для summarization/translation легко займуть кілька ГБ. Переглянути й почистити кеш: `hf cache ls` / `hf cache rm <id моделі>`.

In [34]:
import gc
import torch

for name in ["train_data", "val_data", "data", "token_ids", "token_id_chunks",
             "corpus_text", "words", "unique_words"]:
    globals().pop(name, None)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: зайнято {torch.cuda.memory_allocated() / 1e9:.2f} GB")

### 4.0. Запустіть приклади виконання різних NLP задач

In [35]:
!pip install "transformers[sentencepiece]==5.18.0"

In [36]:
# Sentiment analysis
from transformers import pipeline

classifier = pipeline("sentiment-analysis")
print(classifier("I've been waiting for a HuggingFace course my whole life."))
print(classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
))

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9598048329353333}]
[{'label': 'POSITIVE', 'score': 0.9598048329353333}, {'label': 'NEGATIVE', 'score': 0.9994558691978455}]


In [37]:
# Zero-shot classification
from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.8445985913276672, 0.11197450757026672, 0.04342687129974365]}

In [38]:
# Text generation
from transformers import pipeline

generator = pipeline("text-generation")
print(generator("In this course, we will teach you how to"))

generator = pipeline("text-generation", model="distilgpt2")
print(generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
))

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "In this course, we will teach you how to create a simple web scraper using Python. A web scraper is a tool that extracts data from websites. In this course, we will teach you how to use Python's built-in libraries to scrape data from websites.\nYou will learn:\n- How to use the BeautifulSoup library to parse HTML\n- How to use the requests library to send HTTP requests\n- How to handle errors and exceptions\n- How to store scraped data in a database\n- How to use the pandas library to manipulate and analyze scraped data\n- How to use the sqlite3 library to create and interact with a SQLite database\n- How to use the Scrapy library to create a more advanced web scraper\n- How to use the Selenium library to automate web scraping\n- How to use the Beautiful Soup library to extract data from HTML tables\n- How to use the BeautifulSoup library to extract data from XML\n- How to use the Beautiful Soup library to extract data from JSON\n- How to use the Beautiful Soup lib

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'In this course, we will teach you how to use the tools of the internet to develop a business model for the Internet.\n\n\n\nOne of the things you need to know is that the application of Internet Explorer is to be a web browser, not a web browser. There are many websites to use, but for most companies, the internet is in a very different place than you think.\nThere are also many websites that are not quite Internet Explorer, and you can use them to build a web site.\nThe most interesting thing about the Internet is that this is one of the most popular applications for Windows XP.\nThe application is built on Windows XP, so it is very easy to integrate with Windows XP, but you will need to install the latest version of Windows XP on your computer and use the latest version of Windows XP on your computer.\nTo build a web site for Windows XP, you must have the following steps:\n1. Download the latest version of Windows XP from the Microsoft Store and install the lates

In [39]:
# Fill-mask
from transformers import pipeline

unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

[transformers] No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

[transformers] RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'score': 0.19619713723659515,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical models.'},
 {'score': 0.040527280420064926,
  'token': 38163,
  'token_str': ' computational',
  'sequence': 'This course will teach you all about computational models.'}]

In [40]:
# Named entity recognition
from transformers import pipeline

ner = pipeline("ner", aggregation_strategy="simple")
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'PER',
  'score': np.float32(0.9981694),
  'word': 'Sylvain',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': np.float32(0.9796019),
  'word': 'Hugging Face',
  'start': 33,
  'end': 45},
 {'entity_group': 'LOC',
  'score': np.float32(0.9932106),
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

In [41]:
# Question answering
from transformers import pipeline

question_answerer = pipeline("table-question-answering")
question_answerer(
    query="How many stars does the transformers repository have?",
    table={"Repository":["Transformers","Datasets","Tokenizers"],"Stars":["36542","4512","3934"],"Contributors":["651","77","34"],"Programming language":["Python","Python","Rust, Python and NodeJS"]},
)

[transformers] No model was supplied, defaulted to google/tapas-base-finetuned-wtq and revision e3dde19.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

{'answer': 'AVERAGE > 36542',
 'coordinates': [(0, 1)],
 'cells': ['36542'],
 'aggregator': 'AVERAGE'}

### 4.1. Свої приклади

Повторіть **усі 6 задач** вище (sentiment-analysis, zero-shot-classification, text-generation, fill-mask, NER, table-question-answering), але замість дефолтних англійських прикладів — придумайте й використайте **свої українськомовні приклади** для кожної задачі (якщо потрібно, підберіть відповідні моделі зі знанням української мови). Для виконання цього завдання постворюйте клітинки нижче.

In [42]:
from transformers import pipeline
import gc
import torch

def clear_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("1. Аналіз тональності")
sentiment = pipeline("sentiment-analysis", model="lxyuan/distilbert-base-multilingual-cased-sentiments-student")
print(sentiment("Цей новий аромат від Sisters Aroma просто неймовірний, тримається весь день і дуже приємний"))
print(sentiment("Жахливий сервіс доставки, сайт працює повільно і постійно зависає при оформленні замовлення"))
del sentiment
clear_vram()

print("\n2. Класифікація")
classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")
print(classifier(
    "Компанія оновила дизайн інтернет-магазину та додала зручну фільтрацію товарів",
    candidate_labels=["політика", "електронна комерція", "медицина"]
))
del classifier
clear_vram()

print("\n3. Генерація тексту")
generator = pipeline("text-generation", model="bigscience/bloom-560m")
print(generator("Студент Житомирської політехніки відкрив свій ноутбук щоб", max_new_tokens=30))
del generator
clear_vram()

print("\n4. Заповнення пропусків")
unmasker = pipeline("fill-mask", model="xlm-roberta-base")
print(unmasker("Для створення сучасного веб-додатку найкраще використовувати <mask> та React."))
del unmasker
clear_vram()

print("\n5. Розпізнавання сутностей")
ner = pipeline("ner", model="Davlan/xlm-roberta-base-ner-hrl", aggregation_strategy="simple")
print(ner("Софія має кота на ім'я Бруно."))
del ner
clear_vram()

print("\n6. Відповіді на питання по таблиці")
tqa = pipeline("table-question-answering", model="google/tapas-base-finetuned-wtq")
table = {
    "Бренд": ["Sisters Aroma", "Poetry Home", "Lipss"],
    "Категорія": ["Парфумерія", "Аромати для дому", "Косметика"],
    "Рік": ["2018", "2020", "2019"]
}
print(tqa(query="Який бренд продає аромати для дому?", table=table))
del tqa
clear_vram()

1. Аналіз тональності


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'positive', 'score': 0.8570895791053772}]
[{'label': 'negative', 'score': 0.47669968008995056}]

2. Класифікація


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

{'sequence': 'Компанія оновила дизайн інтернет-магазину та додала зручну фільтрацію товарів', 'labels': ['електронна комерція', 'політика', 'медицина'], 'scores': [0.9691547751426697, 0.030000220984220505, 0.0008449334418401122]}

3. Генерація тексту


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

... (5 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'entity_group': 'PER', 'score': np.float32(0.9943321), 'word': 'Софія', 'start': 0, 'end': 5}, {'entity_group': 'PER', 'score': np.float32(0.9444958), 'word': 'Бруно', 'start': 23, 'end': 28}]

6. Відповіді на питання по таблиці


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

{'answer': 'Sisters Aroma', 'coordinates': [(0, 0)], 'cells': ['Sisters Aroma'], 'aggregator': 'NONE'}


---
## 5. Порівняння моделей

### 5.1. Оберіть задачу й метрику

Візьміть **одну** задачу зі списку нижче:

| Задача | Метрика | Що потрібно підготувати | Метрика в `evaluate` |
|---|---|---|---|
| sentiment-analysis | Accuracy / F1 | 15-20 своїх речень + правильна мітка (positive/negative) | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy), [`f1`](https://huggingface.co/spaces/evaluate-metric/f1) |
| zero-shot-classification | Accuracy | те саме + фіксований список `candidate_labels` | [`accuracy`](https://huggingface.co/spaces/evaluate-metric/accuracy) |
| NER | F1 (entity-level) | кілька речень + список правильних сутностей (текст + тип) | [`seqeval`](https://huggingface.co/spaces/evaluate-metric/seqeval) |

Усі метрики рахуються через бібліотеку `evaluate` (той самий автор, що й `transformers`/`datasets`):
```python
import evaluate

metric = evaluate.load("rouge")  # або "accuracy", "f1", "seqeval", "squad", "sacrebleu"
metric.compute(predictions=[...], references=[...])
```
Формат `predictions`/`references` трохи відрізняється між метриками — дивіться приклад використання на сторінці кожної метрики (посилання в таблиці) або [HF Evaluate — Quick tour](https://huggingface.co/docs/evaluate/a_quick_tour).

### 5.2. Порівняйте моделі

На обраній задачі порівняйте щонайменше 2 різні моделі на однакових вхідних прикладах — не лише значення метрики, а й швидкість інференсу та розмір моделі (кількість параметрів).

### 5.3. Поекспериментуйте з налаштуваннями

Візьміть модель, що показала кращий результат, і поекспериментуйте з її налаштуваннями (temperature/top-p/top-k для генеративних задач, поріг впевненості для класифікаційних задач тощо) — щонайменше 3 зміни, по одній за раз, щоб зрозуміти, що саме впливає на метрику. Зведіть усі спроби (модель × налаштування → значення метрики) в невелику таблицю.

Коротко письмово: яка модель і яка комбінація налаштувань дала найкращий результат за обраною метрикою, і наскільки суттєво налаштування вплинули порівняно з вибором моделі?

In [43]:
!pip install evaluate -q
import time
import evaluate
import pandas as pd
from transformers import pipeline
import gc
import torch

def clear_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

accuracy_metric = evaluate.load("accuracy")

reviews = [
    "Sisters Aroma доставили парфум дуже швидко",
    "Стійкість Poetry Home просто вражає",
    "Lipss прислали помаду в пом'ятій коробці",
    "Кур'єр привіз замовлення вчасно",
    "Аромат дуже швидко вивітрюється",
    "Флакон розбився під час транспортування",
    "Замовлення відправили в день оплати",
    "Крем має дуже приємну текстуру",
    "Коробка була надійно замотана в плівку",
    "Довелося чекати посилку цілий тиждень",
    "Цей парфум викликав алергію на шкірі",
    "Дизайн баночки виглядає дуже преміально",
    "Служба підтримки швидко змінила адресу отримання",
    "Лосьйон чудово зволожує шкіру",
    "Дозатор на пляшці виявився зламаним"
]

candidate_labels = ["доставка", "якість", "пакування"]
true_labels = [
    "доставка", "якість", "пакування", "доставка", "якість",
    "пакування", "доставка", "якість", "пакування", "доставка",
    "якість", "пакування", "доставка", "якість", "пакування"
]

label2id = {"доставка": 0, "якість": 1, "пакування": 2}
true_labels_ids = [label2id[label] for label in true_labels]

def run_eval(model_name, template=None):
    pipe = pipeline("zero-shot-classification", model=model_name)
    start_t = time.time()
    preds = []
    for r in reviews:
        if template:
            res = pipe(r, candidate_labels=candidate_labels, hypothesis_template=template)
        else:
            res = pipe(r, candidate_labels=candidate_labels)
        preds.append(res["labels"][0])
    end_t = time.time()

    preds_ids = [label2id[p] for p in preds]
    acc = accuracy_metric.compute(predictions=preds_ids, references=true_labels_ids)["accuracy"]

    del pipe
    clear_mem()
    return acc, end_t - start_t

print("Тестуємо модель 1 mDeBERTa-v3")
acc1, t1 = run_eval("MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

print("Тестуємо модель 2 xlm-roberta")
acc2, t2 = run_eval("symanto/xlm-roberta-base-snli-mnli-anli-xnli")

print("Експеримент 1 Шаблон про відгук")
acc3, t3 = run_eval("MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", "Цей відгук про {}.")

print("Експеримент 2 Шаблон про категорію")
acc4, t4 = run_eval("MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", "Категорія цього тексту {}.")

print("Експеримент 3 Шаблон про клієнта")
acc5, t5 = run_eval("MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", "Клієнт оцінює {}.")

results = [
    {"Модель": "mDeBERTa-v3", "Налаштування": "Стандартний шаблон", "Точність": acc1, "Час (с)": round(t1, 2)},
    {"Модель": "xlm-roberta", "Налаштування": "Стандартний шаблон", "Точність": acc2, "Час (с)": round(t2, 2)},
    {"Модель": "mDeBERTa-v3", "Налаштування": "Шаблон відгук", "Точність": acc3, "Час (с)": round(t3, 2)},
    {"Модель": "mDeBERTa-v3", "Налаштування": "Шаблон категорія", "Точність": acc4, "Час (с)": round(t4, 2)},
    {"Модель": "mDeBERTa-v3", "Налаштування": "Шаблон клієнт", "Точність": acc5, "Час (с)": round(t5, 2)}
]

display(pd.DataFrame(results))

Тестуємо модель 1 mDeBERTa-v3


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Тестуємо модель 2 xlm-roberta


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Експеримент 1 Шаблон про відгук


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Експеримент 2 Шаблон про категорію


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Експеримент 3 Шаблон про клієнта


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

,Модель,Налаштування,Точність,Час (с)
0,mDeBERTa-v3,Стандартний шаблон,0.933333,140.50
1,xlm-roberta,Стандартний шаблон,0.666667,9.82
2,mDeBERTa-v3,Шаблон відгук,0.866667,132.75
3,mDeBERTa-v3,Шаблон категорія,0.666667,133.82
4,mDeBERTa-v3,Шаблон клієнт,0.400000,130.78


За результатами порівняння модель mDeBERTa показала найвищу точність класифікації із базовим налаштуванням і досягла понад дев'яноста трьох відсотків. Друга модель виявилася значно швидшою але її точність склала лише шістдесят шість відсотків. Експерименти зі зміною текстового шаблону показали несподіваний результат адже всі власні україномовні варіанти погіршили початкову метрику. Найгірше спрацював шаблон про клієнта який знизив точність до сорока відсотків що свідчить про високу чутливість моделі до контексту та синтаксису запиту.

---
## 6. Творчий застосунок

Побудуйте невеликий Gradio-застосунок на основі будь-якої готової моделі з розділів 4-5 — тема й формат повністю на ваш вибір, заохочується творчість і щось не зовсім очевидне (не просто "ще один чат-бот"). Це може бути й генеративна LLM через `transformers` (`AutoModelForCausalLM`/`AutoTokenizer` або `pipeline("text-generation", ...)`, той самий патерн, що й у демо 4.1-4.2 на Qwen) — `text-generation` уже входить до обов'язкових 8 задач розділу 4, тож це не вихід за межі завдання. Головне — щоб застосунок реально запускався і демонстрував конкретну, зрозумілу користувацьку цінність.

In [9]:
!pip install gradio -q
import gradio as gr
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")
labels = ["доставка", "якість товару", "пакування", "робота консультанта", "повернення коштів"]

def analyze_ticket(text):
    result = classifier(text, candidate_labels=labels)
    best_label = result["labels"][0]
    score = result["scores"][0]
    return f"Відділ {best_label}", f"Впевненість {score:.1%}"

demo = gr.Interface(
    fn=analyze_ticket,
    inputs=gr.Textbox(lines=4, placeholder="Введіть текст звернення клієнта..."),
    outputs=[gr.Textbox(label="Категорія звернення"), gr.Textbox(label="Впевненість нейромережі")],
    title="Сортувальник звернень для e-commerce",
    description="Цей застосунок автоматично визначає категорію клієнтського запиту щоб передати його у відповідний відділ інтернет-магазину"
)

demo.launch(share=True)

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 4.31MB            

... (2 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://339316b6a32bde6bd7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [12]:
import pandas as pd

final_data = [
    ["minigpt_v1", "val_loss", 5.194],
    ["minigpt_v2", "val_loss", 5.262],
    ["mDeBERTa-v3_standart", "accuracy", 0.933],
    ["xlm-roberta_standart", "accuracy", 0.666],
    ["mDeBERTa-v3_template_review", "accuracy", 0.866],
    ["mDeBERTa-v3_template_category", "accuracy", 0.666],
    ["mDeBERTa-v3_template_client", "accuracy", 0.400]
]

df_results = pd.DataFrame(final_data, columns=["Model", "Metric_Type", "Score"])
df_results.to_csv(os.path.join(LAB_DIR, "results.csv"), index=False)
print(df_results)

                           Model Metric_Type  Score
0                     minigpt_v1    val_loss  5.194
1                     minigpt_v2    val_loss  5.262
2           mDeBERTa-v3_standart    accuracy  0.933
3           xlm-roberta_standart    accuracy  0.666
4    mDeBERTa-v3_template_review    accuracy  0.866
5  mDeBERTa-v3_template_category    accuracy  0.666
6    mDeBERTa-v3_template_client    accuracy  0.400


---
### Як здавати лабу

У [форму](https://forms.gle/Z1gxmsDs656jqsDGA) здати посилання на **публічний** репозиторій, який містить:

1. **Демо-ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
2. **Цей ноутбук з усіма виконаними пунктами**. Має запускатися зверху до низу (Run All) в будь-якому середовищі без помилок.
3. **Файли всіх натренованих варіантів mini-GPT** (3.2) — кожен окремим `.pt` файлом.
4. **Логи TensorBoard** — папка `runs/` з усіма прогонами (і mini-GPT, і, за бажання, логи порівняння в розділі 5).
5. **Таблицю результатів** — `results.csv` з порівнянням варіантів mini-GPT (val_loss) і моделей з розділу 5.
6. Структура файлів на репозиторії, **точно дотримуйтесь, інакше не зарахується лаба**:
```
/lab3/transformers_demo.ipynb
/lab3/transformers_task.ipynb
/lab3/models/minigpt_v1.pt
/lab3/models/minigpt_v2.pt
/lab3/models/...        (усі інші варіанти з 3.2)
/lab3/runs/*
/lab3/results.csv
```